In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Burgers equation parameters
nu = 0.01 / np.pi

# Domain
x_min, x_max = -1.0, 1.0
t_min, t_max = 0.0, 1.0

# Initial condition
def initial_condition(x):
    return -np.sin(np.pi * x)

# Number of training points
N_f = 5000   # PDE collocation points
N_ic = 200   # Initial-condition points
N_bc = 200   # Boundary-condition points

# PDE collocation points
x_f = np.random.uniform(x_min, x_max, (N_f, 1))
t_f = np.random.uniform(t_min, t_max, (N_f, 1))

# Initial-condition points
x_ic = np.linspace(x_min, x_max, N_ic).reshape(-1, 1)
t_ic = np.zeros_like(x_ic)
u_ic = initial_condition(x_ic)

# Boundary points
t_bc = np.linspace(t_min, t_max, N_bc).reshape(-1, 1)

x_bc_left = np.full_like(t_bc, x_min)
x_bc_right = np.full_like(t_bc, x_max)

u_bc_left = np.zeros_like(t_bc)
u_bc_right = np.zeros_like(t_bc)

print("PDE points:", x_f.shape)
print("IC points:", x_ic.shape)
print("BC points:", t_bc.shape)


In [ ]:
# 1. Plot initial condition
plt.figure(figsize=(7, 4))
plt.plot(x_ic, u_ic, label="Initial condition")
plt.xlabel("x")
plt.ylabel("u(x,0)")
plt.title("Initial Condition")
plt.grid()
plt.legend()
plt.show()


# 2. Plot PDE collocation points
plt.figure(figsize=(7, 4))
plt.scatter(x_f, t_f, s=2)
plt.xlabel("x")
plt.ylabel("t")
plt.title("PDE Collocation Points")
plt.grid()
plt.show()


# 3. Plot boundary points
plt.figure(figsize=(7, 4))
plt.scatter(x_bc_left, t_bc, s=5, label="Left boundary")
plt.scatter(x_bc_right, t_bc, s=5, label="Right boundary")
plt.xlabel("x")
plt.ylabel("t")
plt.title("Boundary Points")
plt.grid()
plt.legend()
plt.show()


In [ ]:
# ============================================================
# STEP 3: BUILD THE MLP
# Purpose:
# Create the neural network u_theta(x,t) that will approximate
# the unknown solution u(x,t) of the Burgers equation.
#
# Input  : (x, t)
# Output : predicted solution u_theta(x,t)
# ============================================================

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 1)
        )

    def forward(self, x, t):
        # Combine x and t into a single input
        inputs = torch.cat([x, t], dim=1)

        # Predict u_theta(x,t)
        return self.network(inputs)


# Create the MLP and move it to the selected device
model = MLP().to(device)

print(model)


In [ ]:
# ============================================================
# STEP 4: PREPARE TRAINING POINTS FOR AUTOMATIC DIFFERENTIATION
# Purpose:
# Convert our NumPy training points into PyTorch tensors.
# requires_grad=True allows PyTorch to calculate derivatives
# of the neural-network output with respect to x and t.
# ============================================================

# PDE collocation points
x_f_t = torch.tensor(x_f, dtype=torch.float32, device=device, requires_grad=True)
t_f_t = torch.tensor(t_f, dtype=torch.float32, device=device, requires_grad=True)

# Initial-condition points
x_ic_t = torch.tensor(x_ic, dtype=torch.float32, device=device)
t_ic_t = torch.tensor(t_ic, dtype=torch.float32, device=device)
u_ic_t = torch.tensor(u_ic, dtype=torch.float32, device=device)

# Boundary-condition points
x_bc_left_t = torch.tensor(
    x_bc_left, dtype=torch.float32, device=device
)
x_bc_right_t = torch.tensor(
    x_bc_right, dtype=torch.float32, device=device
)
t_bc_t = torch.tensor(
    t_bc, dtype=torch.float32, device=device
)

# Check the tensor shapes
print("PDE x:", x_f_t.shape)
print("PDE t:", t_f_t.shape)
print("IC x:", x_ic_t.shape)
print("BC t:", t_bc_t.shape)


In [ ]:
# ============================================================
# STEP 5: COMPUTE DERIVATIVES USING AUTOMATIC DIFFERENTIATION
# Purpose:
# Calculate u_t, u_x, and u_xx from the neural-network
# prediction u_theta(x,t) using PyTorch autograd.
# ============================================================

def compute_derivatives_AD(model, x, t):

    # Neural-network prediction
    u = model(x, t)

    # First derivatives
    u_x = torch.autograd.grad(
        u,
        x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    u_t = torch.autograd.grad(
        u,
        t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    # Second derivative with respect to x
    u_xx = torch.autograd.grad(
        u_x,
        x,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0]

    return u, u_x, u_t, u_xx


# Test the derivative calculation
u, u_x, u_t, u_xx = compute_derivatives_AD(
    model,
    x_f_t,
    t_f_t
)

print("u shape   :", u.shape)
print("u_x shape :", u_x.shape)
print("u_t shape :", u_t.shape)
print("u_xx shape:", u_xx.shape)


In [ ]:
# ============================================================
# STEP 6: CONSTRUCT PDE RESIDUAL AND PINN LOSS
# Purpose:
# Use the automatic derivatives to construct the Burgers
# equation residual and the initial/boundary-condition losses.
# The total PINN loss is:
# L = L_PDE + L_IC + L_BC
# ============================================================

def compute_losses(model):

    # --------------------------------------------------------
    # 1. PDE loss
    # --------------------------------------------------------

    u, u_x, u_t, u_xx = compute_derivatives_AD(
        model, x_f_t, t_f_t
    )

    # Burgers equation residual
    residual = u_t + u * u_x - nu * u_xx

    # Mean squared PDE residual
    loss_pde = torch.mean(residual**2)

    # --------------------------------------------------------
    # 2. Initial-condition loss
    # --------------------------------------------------------

    u_pred_ic = model(x_ic_t, t_ic_t)

    loss_ic = torch.mean((u_pred_ic - u_ic_t)**2)

    # --------------------------------------------------------
    # 3. Boundary-condition loss
    # --------------------------------------------------------

    u_pred_left = model(x_bc_left_t, t_bc_t)
    u_pred_right = model(x_bc_right_t, t_bc_t)

    loss_bc = torch.mean(u_pred_left**2) + \
              torch.mean(u_pred_right**2)

    # --------------------------------------------------------
    # Total loss
    # --------------------------------------------------------

    total_loss = loss_pde + loss_ic + loss_bc

    return total_loss, loss_pde, loss_ic, loss_bc


# Test the loss calculation
losses = compute_losses(model)

print("Total loss :", losses[0].item())
print("PDE loss   :", losses[1].item())
print("IC loss    :", losses[2].item())
print("BC loss    :", losses[3].item())


In [ ]:
# ============================================================
# STEP 7: TRAIN THE AD-PINN
# Purpose:
# Train the MLP so that its prediction satisfies the Burgers
# PDE, the initial condition, and the boundary conditions.
# ============================================================

import time

# Create a fresh model for the AD-PINN
ad_model = MLP().to(device)

# Adam optimizer
optimizer = torch.optim.Adam(
    ad_model.parameters(),
    lr=1e-3
)

# Number of training iterations
epochs = 10000

# Store loss values for convergence plot
loss_history = []

# Start timing
start_time = time.time()

for epoch in range(epochs):

    # Clear previous gradients
    optimizer.zero_grad()

    # Calculate PINN losses
    total_loss, loss_pde, loss_ic, loss_bc = compute_losses(ad_model)

    # Backpropagation
    total_loss.backward()

    # Update network parameters
    optimizer.step()

    # Store total loss
    loss_history.append(total_loss.item())

    # Print progress
    if (epoch + 1) % 1000 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {total_loss.item():.6e}"
        )

# Training time
training_time = time.time() - start_time

print("\nAD-PINN training completed.")
print(f"Training time: {training_time:.2f} seconds")


In [ ]:
# ============================================================
# STEP 8: CHECK AD-PINN CONVERGENCE
# Purpose:
# Plot the training loss to see whether the AD-PINN
# converged during the 10,000 training iterations.
# ============================================================

plt.figure(figsize=(7, 4))

plt.semilogy(loss_history)

plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("AD-PINN Training Loss")
plt.grid()

plt.show()


In [ ]:
# ============================================================
# STEP 9: GENERATE THE REFERENCE NUMERICAL SOLUTION
# Purpose:
# Solve the Burgers equation independently using a conventional
# numerical method.
#
# This reference solution is NOT used to train the PINN.
# It will only be used later to measure prediction accuracy.
# ============================================================

import numpy as np
from scipy.integrate import solve_ivp

# ------------------------------------------------------------
# 1. Reference grid
# ------------------------------------------------------------

Nx_ref = 256
Nt_ref = 101

x_ref = np.linspace(x_min, x_max, Nx_ref)
t_ref = np.linspace(t_min, t_max, Nt_ref)

dx = x_ref[1] - x_ref[0]

print("dx =", dx)


# ------------------------------------------------------------
# 2. Initial condition
# ------------------------------------------------------------

u0 = initial_condition(x_ref)

# Interior points only
u0_interior = u0[1:-1]


# ------------------------------------------------------------
# 3. Define the spatially discretized Burgers equation
# ------------------------------------------------------------

def burgers_rhs(t, u_interior):

    # Create full spatial vector
    u = np.zeros(Nx_ref)

    # Insert interior values
    u[1:-1] = u_interior

    # Neighboring values
    u_left = u[:-2]
    u_center = u[1:-1]
    u_right = u[2:]

    # --------------------------------------------------------
    # Upwind approximation for u_x
    # --------------------------------------------------------

    u_x_backward = (u_center - u_left) / dx
    u_x_forward = (u_right - u_center) / dx

    # Choose direction according to the sign of u
    u_x = np.where(
        u_center >= 0,
        u_x_backward,
        u_x_forward
    )

    # --------------------------------------------------------
    # Central difference for u_xx
    # --------------------------------------------------------

    u_xx = (
        u_right
        - 2.0 * u_center
        + u_left
    ) / (dx ** 2)

    # --------------------------------------------------------
    # Burgers equation
    #
    # u_t + u*u_x - nu*u_xx = 0
    #
    # Therefore:
    #
    # u_t = -u*u_x + nu*u_xx
    # --------------------------------------------------------

    du_dt = -u_center * u_x + nu * u_xx

    return du_dt


# ------------------------------------------------------------
# 4. Solve the discretized system
# ------------------------------------------------------------

solution = solve_ivp(
    burgers_rhs,
    (t_min, t_max),
    u0_interior,
    t_eval=t_ref,
    method="BDF",
    rtol=1e-6,
    atol=1e-8
)


# ------------------------------------------------------------
# 5. Check whether the solver completed successfully
# ------------------------------------------------------------

print("Solver successful:", solution.success)
print("Solver message:", solution.message)

if not solution.success:
    raise RuntimeError(
        "Reference solver failed. Do not continue to evaluation."
    )


# ------------------------------------------------------------
# 6. Construct the complete reference solution
# ------------------------------------------------------------

u_ref = np.zeros((Nt_ref, Nx_ref))

# Interior solution
u_ref[:, 1:-1] = solution.y.T

# Dirichlet boundary conditions
u_ref[:, 0] = 0.0
u_ref[:, -1] = 0.0


# ------------------------------------------------------------
# 7. Basic checks
# ------------------------------------------------------------

print("Reference solution shape:", u_ref.shape)
print("Minimum u:", u_ref.min())
print("Maximum u:", u_ref.max())# ============================================================
# STEP 10: VISUALIZE THE REFERENCE SOLUTION
# Purpose:
# Plot the numerical/reference solution of the Burgers equation
# over the complete (x,t) domain.
# ============================================================

plt.figure(figsize=(8, 5))

plt.imshow(
    u_ref,
    extent=[x_min, x_max, t_min, t_max],
    origin="lower",
    aspect="auto"
)

plt.colorbar(label="u(x,t)")

plt.xlabel("x")
plt.ylabel("t")
plt.title("Reference Solution of 1D Burgers Equation")

plt.show()


In [ ]:
# ============================================================
# STEP 10: VISUALIZE THE REFERENCE SOLUTION
# Purpose:
# Plot the numerical/reference solution of the Burgers equation
# over the complete (x,t) domain.
# ============================================================

plt.figure(figsize=(8, 5))

plt.imshow(
    u_ref,
    extent=[x_min, x_max, t_min, t_max],
    origin="lower",
    aspect="auto"
)

plt.colorbar(label="u(x,t)")

plt.xlabel("x")
plt.ylabel("t")
plt.title("Reference Solution of 1D Burgers Equation")

plt.show()


In [ ]:
# ============================================================
# STEP 11: GENERATE AD-PINN PREDICTIONS
# Purpose:
# Evaluate the trained AD-PINN on the same (x,t) grid used
# by the reference numerical solution.
# ============================================================

# Create a mesh of x and t points
X, T = np.meshgrid(x_ref, t_ref)

# Flatten the grid
X_flat = X.reshape(-1, 1)
T_flat = T.reshape(-1, 1)

# Convert to PyTorch tensors
X_tensor = torch.tensor(
    X_flat,
    dtype=torch.float32,
    device=device
)

T_tensor = torch.tensor(
    T_flat,
    dtype=torch.float32,
    device=device
)

# We are only predicting, so gradients are not needed
with torch.no_grad():

    u_ad = ad_model(
        X_tensor,
        T_tensor
    ).cpu().numpy()

# Reshape back to (time, space)
u_ad = u_ad.reshape(Nt_ref, Nx_ref)

print("AD-PINN prediction shape:", u_ad.shape)
print("AD-PINN minimum:", u_ad.min())
print("AD-PINN maximum:", u_ad.max())


In [ ]:
# ============================================================
# STEP 12: CALCULATE RELATIVE L2 ERROR
# Purpose:
# Measure how close the trained AD-PINN solution is to the
# independent reference numerical solution.
# ============================================================

# Difference between AD-PINN and reference solution
error = u_ad - u_ref

# Relative L2 error
relative_l2_error = (
    np.linalg.norm(error)
    / np.linalg.norm(u_ref)
)

print(f"Relative L2 Error: {relative_l2_error:.6e}")


In [ ]:
# ============================================================
# STEP 13: VISUAL COMPARISON
# Purpose:
# Compare the AD-PINN prediction with the independent
# reference solution.
# ============================================================

# Reference solution
plt.figure(figsize=(8, 5))

plt.imshow(
    u_ref,
    extent=[x_min, x_max, t_min, t_max],
    origin="lower",
    aspect="auto"
)

plt.colorbar(label="u(x,t)")
plt.xlabel("x")
plt.ylabel("t")
plt.title("Reference Solution")

plt.show()


# AD-PINN solution
plt.figure(figsize=(8, 5))

plt.imshow(
    u_ad,
    extent=[x_min, x_max, t_min, t_max],
    origin="lower",
    aspect="auto"
)

plt.colorbar(label="u(x,t)")
plt.xlabel("x")
plt.ylabel("t")
plt.title("AD-PINN Solution")

plt.show()


In [ ]:
# ============================================================
# STEP 22A: FDM-SAFE PDE COLLOCATION POINTS
# Purpose:
# Create interior collocation points so that the central
# finite-difference stencil remains inside the computational
# domain.
# ============================================================

# FDM step size used for the baseline experiment
FDM_STEP_SIZE = 1e-2

# Generate FDM collocation points away from the boundaries
x_f_FDM = np.random.uniform(
    x_min + FDM_STEP_SIZE,
    x_max - FDM_STEP_SIZE,
    (N_f, 1)
)

t_f_FDM = np.random.uniform(
    t_min + FDM_STEP_SIZE,
    t_max - FDM_STEP_SIZE,
    (N_f, 1)
)

# Convert the FDM collocation points to PyTorch tensors
x_f_FDM_t = torch.tensor(
    x_f_FDM,
    dtype=torch.float32,
    device=device
)

t_f_FDM_t = torch.tensor(
    t_f_FDM,
    dtype=torch.float32,
    device=device
)

print("FDM collocation points created.")
print("x range:", x_f_FDM.min(), "to", x_f_FDM.max())
print("t range:", t_f_FDM.min(), "to", t_f_FDM.max())


In [ ]:
# ============================================================
# STEP 22C: FDM DERIVATIVE FUNCTION
# Purpose:
# Define the finite-difference derivative calculations again
# so the h-step-size study can use them.
# ============================================================

def compute_derivatives_FDM(model, x, t, h):
    
    # Network prediction at the original point
    u = model(x, t)

    # --------------------------------------------------------
    # FDM: x-direction evaluations
    # --------------------------------------------------------
    u_x_plus = model(x + h, t)
    u_x_minus = model(x - h, t)

    # --------------------------------------------------------
    # FDM: t-direction evaluations
    # --------------------------------------------------------
    u_t_plus = model(x, t + h)
    u_t_minus = model(x, t - h)

    # --------------------------------------------------------
    # Central finite difference for u_x
    # --------------------------------------------------------
    u_x = (u_x_plus - u_x_minus) / (2 * h)

    # --------------------------------------------------------
    # Central finite difference for u_t
    # --------------------------------------------------------
    u_t = (u_t_plus - u_t_minus) / (2 * h)

    # --------------------------------------------------------
    # Central finite difference for u_xx
    # --------------------------------------------------------
    u_xx = (
        u_x_plus - 2 * u + u_x_minus
    ) / (h ** 2)

    return u, u_x, u_t, u_xx


In [ ]:
# ============================================================
# STEP 22C + STEP 23:
# FDM-PINN + FDM STEP-SIZE (h) STUDY
#
# Purpose:
# Investigate how the finite-difference step size h affects
# FDM-PINN accuracy, PDE residual, and training time.
# ============================================================

import time
import numpy as np
import torch

# ============================================================
# FDM LOSS FUNCTION
# ============================================================

def compute_losses_FDM_safe(model, h):

    # --------------------------------------------------------
    # 1. FDM PDE collocation points
    # --------------------------------------------------------
    x = x_f_FDM_t
    t = t_f_FDM_t

    # --------------------------------------------------------
    # 2. Network prediction at original points
    # --------------------------------------------------------
    u = model(x, t)

    # --------------------------------------------------------
    # 3. Finite-difference points in x-direction
    # --------------------------------------------------------
    u_x_plus = model(x + h, t)
    u_x_minus = model(x - h, t)

    # --------------------------------------------------------
    # 4. Finite-difference points in t-direction
    # --------------------------------------------------------
    u_t_plus = model(x, t + h)
    u_t_minus = model(x, t - h)

    # --------------------------------------------------------
    # 5. First derivative du/dx
    # --------------------------------------------------------
    u_x = (u_x_plus - u_x_minus) / (2.0 * h)

    # --------------------------------------------------------
    # 6. First derivative du/dt
    # --------------------------------------------------------
    u_t = (u_t_plus - u_t_minus) / (2.0 * h)

    # --------------------------------------------------------
    # 7. Second derivative d2u/dx2
    # --------------------------------------------------------
    u_xx = (
        u_x_plus
        - 2.0 * u
        + u_x_minus
    ) / (h ** 2)

    # --------------------------------------------------------
    # 8. Burgers equation
    #
    # u_t + u*u_x - nu*u_xx = 0
    # --------------------------------------------------------
    residual = u_t + u * u_x - nu * u_xx

    loss_pde = torch.mean(residual ** 2)

    # ========================================================
    # 9. Initial Condition
    # ========================================================

    u_pred_ic = model(
        x_ic_t,
        t_ic_t
    )

    loss_ic = torch.mean(
        (u_pred_ic - u_ic_t) ** 2
    )

    # ========================================================
    # 10. Boundary Condition
    # ========================================================

    u_pred_left = model(
        x_bc_left_t,
        t_bc_t
    )

    u_pred_right = model(
        x_bc_right_t,
        t_bc_t
    )

    loss_bc = (
        torch.mean(u_pred_left ** 2)
        +
        torch.mean(u_pred_right ** 2)
    )

    # ========================================================
    # 11. Total PINN Loss
    # ========================================================

    total_loss = (
        loss_pde
        + loss_ic
        + loss_bc
    )

    return (
        total_loss,
        loss_pde,
        loss_ic,
        loss_bc
    )


# ============================================================
# TEST THE FDM LOSS FUNCTION
# ============================================================

test_model = MLP().to(device)

test_loss = compute_losses_FDM_safe(
    test_model,
    1e-2
)

print("FDM loss function test successful.")
print(f"Total Loss : {test_loss[0].item():.6e}")
print(f"PDE Loss   : {test_loss[1].item():.6e}")
print(f"IC Loss    : {test_loss[2].item():.6e}")
print(f"BC Loss    : {test_loss[3].item():.6e}")


# ============================================================
# STEP 23: FDM STEP-SIZE STUDY
# ============================================================

# Step sizes to investigate
h_values = [
    1e-1,
    1e-2,
    1e-3
]

# Store results
h_results = []

# Number of training epochs
h_study_epochs = 5000


# ============================================================
# TRAIN ONE MODEL FOR EACH h
# ============================================================

for h_value in h_values:

    print("\n" + "=" * 60)
    print(
        f"Training FDM-PINN with h = {h_value:.0e}"
    )
    print("=" * 60)

    # --------------------------------------------------------
    # Fresh model
    # --------------------------------------------------------
    h_model = MLP().to(device)

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------
    h_optimizer = torch.optim.Adam(
        h_model.parameters(),
        lr=1e-3
    )

    # Loss history
    h_loss_history = []

    # Start timer
    h_start_time = time.time()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    for epoch in range(h_study_epochs):

        h_optimizer.zero_grad()

        (
            total_loss,
            loss_pde,
            loss_ic,
            loss_bc
        ) = compute_losses_FDM_safe(
            h_model,
            h_value
        )

        # Backpropagation
        total_loss.backward()

        # Parameter update
        h_optimizer.step()

        # Store loss
        h_loss_history.append(
            total_loss.item()
        )

        # Progress
        if (epoch + 1) % 1000 == 0:

            print(
                f"Epoch [{epoch + 1}/{h_study_epochs}] "
                f"Loss: {total_loss.item():.6e}"
            )

    # --------------------------------------------------------
    # Training time
    # --------------------------------------------------------
    h_training_time = (
        time.time() - h_start_time
    )

    # ========================================================
    # Prediction on reference grid
    # ========================================================

    with torch.no_grad():

        u_h = h_model(
            X_tensor,
            T_tensor
        ).cpu().numpy()

    # Reshape prediction
    u_h = u_h.reshape(
        Nt_ref,
        Nx_ref
    )

    # ========================================================
    # Relative L2 Error
    # ========================================================

    h_error = u_h - u_ref

    h_relative_l2 = (
        np.linalg.norm(h_error)
        /
        np.linalg.norm(u_ref)
    )

    # ========================================================
    # PDE Residual
    # ========================================================

    with torch.no_grad():

        _, h_pde_loss, _, _ = (
            compute_losses_FDM_safe(
                h_model,
                h_value
            )
        )

    # ========================================================
    # Store Results
    # ========================================================

    h_results.append({

        "h": h_value,

        "relative_l2_error":
            h_relative_l2,

        "pde_residual":
            h_pde_loss.item(),

        "training_time":
            h_training_time

    })

    # ========================================================
    # Print Results
    # ========================================================

    print("\nResults:")

    print(
        f"Relative L2 Error : "
        f"{h_relative_l2:.6e}"
    )

    print(
        f"PDE Residual MSE  : "
        f"{h_pde_loss.item():.6e}"
    )

    print(
        f"Training Time     : "
        f"{h_training_time:.2f} seconds"
    )


# ============================================================
# FINAL RESULTS TABLE
# ============================================================

print("\n")
print("=" * 75)
print("FDM STEP-SIZE STUDY RESULTS")
print("=" * 75)

print(
    f"{'h':<12}"
    f"{'Relative L2 Error':<25}"
    f"{'PDE Residual':<25}"
    f"{'Time (s)':<15}"
)

print("-" * 75)

for result in h_results:

    print(
        f"{result['h']:<12.0e}"
        f"{result['relative_l2_error']:<25.6e}"
        f"{result['pde_residual']:<25.6e}"
        f"{result['training_time']:<15.2f}"
    )

print("=" * 75)
